In [1]:
from lib.hw_build import HwBuildHelper
from lib.sw_build import SwBuildHelper

In [2]:
# ── Streamer definitions ────────────────────────────────────────────────────
# Index 0 is always the DMA pass-through streamer.
# Each entry: load_width / store_width in bytes (must be power of two),
#             actual_width in bits (<= load/store_width * 8),
#             amount_row: BRAM depth.

dfx_streamers_meta = [
    # streamer 0
    {"bank_req_per_set"   : 1,      # amount bank used in each set (cannot lower or higher )
     "amount_set"         : 1,      # amount of replication of bank set
     "amount_row_per_set" : 4096,   # row per set
     "width_per_bank"     : 32//8       # width in byte
     },
    ################### store from region 0
    # streamer 1
    {"bank_req_per_set"   : 4,
     "amount_set"         : 4,   # 16 bank in total 2*8
     "amount_row_per_set" : 4096,
     "width_per_bank"     : 8    # uram each bank have
     },
    # steramer 2
    {"bank_req_per_set"   : 2,
     "amount_set"         : 15,   # 16 bank in total 4*4
     "amount_row_per_set" : 4096,
     "width_per_bank"     : 8    # uram each bank have
     },
    ################### store from region 1
    # steramer 3
    {"bank_req_per_set"   : 2,
     "amount_set"         : 1,   # 16 bank in total 4*4
     "amount_row_per_set" : 4096,
     "width_per_bank"     : 8    # uram each bank have
     },
    # steramer 4
    {"bank_req_per_set"   : 4,
     "amount_set"         : 4,   # 16 bank in total 4*4
     "amount_row_per_set" : 4096,
     "width_per_bank"     : 8    # uram each bank have
     },
]

def build_dfx_streamers(meta_list):
    """Build dfx_streamers from dfx_streamers_meta.

    For each entry:
        load_width   = bank_req_per_set * width_per_bank   (bytes)
        store_width  = load_width
        actual_width = load_width * 8                      (bits)
        amount_row   = amount_set * amount_row_per_set
    """
    streamers = []
    for m in meta_list:
        load_width   = m["bank_req_per_set"] * m["width_per_bank"]
        store_width  = load_width
        actual_width = load_width * 8
        amount_row   = m["amount_set"] * m["amount_row_per_set"]
        streamers.append({
            "load_width"  : load_width,
            "store_width" : store_width,
            "actual_width": actual_width,
            "amount_row"  : amount_row,
        })
    return streamers

dfx_streamers = build_dfx_streamers(dfx_streamers_meta)
for i, s in enumerate(dfx_streamers):
    print(f"  streamer {i}: {s}")

# ── Region definitions ──────────────────────────────────────────────────────
# Two reconfigurable regions forming a pipeline: DMA → region 0 (RM0) → region 1 (RM0) → region 0 (RM1) → region 1 (RM1) → DMA
# load_streamers / store_streamers: list of streamer indices connected to this region.
dfx_regions = [
    {"load_streamers": [0, 3, 4]   , "store_streamers": [1, 2]   },
    {"load_streamers": [1, 2], "store_streamers": [0, 3, 4]},
]

# ── Reconfigurable module (RM) schematics ───────────────────────────────────
# 2-D list: rm_schemetics[region_idx][rm_idx]
# load_io_map / store_io_map: list of (streamer_index, kernel_port_index) pairs.
# kernel_port_index is fixed to 0 for all entries.
# All (load_streamer x store_streamer) combinations are enumerated per region.
#
# Region 0: load from [s0, s3, s4] x store to [s1, s2] → 3x2 = 6 RMs
# Region 1: load from [s1, s2]     x store to [s0, s3, s4] → 2x3 = 6 RMs
rm_schemetics = [
    [  # region 0: load_streamers=[s0,s3,s4], store_streamers=[s1,s2]
        {"load_io_map": [], "store_io_map": []},  # rm_0
        {"load_io_map": [], "store_io_map": []},  # rm_1
    ],
    [  # region 1: load_streamers=[s1,s2], store_streamers=[s0,s3,s4]
        {"load_io_map": [], "store_io_map": []},  # rm_0
        {"load_io_map": [], "store_io_map": []},  # rm_1
    ],
]

# ── Instantiate HwBuildHelper ───────────────────────────────────────────────
hw_builder = HwBuildHelper(
    build_folder_path="./build_prj",
    dfx_root_path=".",
    board="kv260",
    user_repo_path="",
    user_rm_build_tcl_path="",
    req_gen_ip=1,
    num_core=4,
    clk_frq=99999001,          # Hz
    rm_index_width=3,           # 1 << rm_index_width = max bank-1 slots; 6 RMs < 8
    dfx_streamers=dfx_streamers,
    dfx_regions=dfx_regions,
    rm_schemetics=rm_schemetics,
    test_mode=1,
    vivado_path="/tools/Xilinx/Vivado/2023.2/bin/vivado",
    export_folder_path="./export"
)

  streamer 0: {'load_width': 4, 'store_width': 4, 'actual_width': 32, 'amount_row': 4096}
  streamer 1: {'load_width': 32, 'store_width': 32, 'actual_width': 256, 'amount_row': 16384}
  streamer 2: {'load_width': 16, 'store_width': 16, 'actual_width': 128, 'amount_row': 61440}
  streamer 3: {'load_width': 16, 'store_width': 16, 'actual_width': 128, 'amount_row': 4096}
  streamer 4: {'load_width': 32, 'store_width': 32, 'actual_width': 256, 'amount_row': 16384}


In [3]:
hw_builder.run_build()


Running Vivado with /media/tanawin/tanawin1701e/project8/dfx4ml/dfx4ml_code/build_prj/run_build.tcl...

****** Vivado v2023.2 (64-bit)
  **** SW Build 4029153 on Fri Oct 13 20:13:54 MDT 2023
  **** IP Build 4028589 on Sat Oct 14 00:45:43 MDT 2023
  **** SharedData Build 4025554 on Tue Oct 10 17:18:54 MDT 2023
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2023 Advanced Micro Devices, Inc. All Rights Reserved.

start_gui
INFO: [Common 17-206] Exiting Vivado at Thu Jun 11 17:35:00 2026...


In [ ]:
hw_builder.package_export_files()

In [ ]:
# All parameters (export_folder_path, num_pr_region, rm_index_width, num_streamer)
# are derived from hw_builder; pass any of them explicitly to override.
sw_builder = SwBuildHelper(hw_builder=hw_builder)

In [ ]:
sw_builder.package_export_file()